# GuideSep — mask-free inference

Place `wav/guide.wav` and `wav/mix.wav`, then run all cells. The separated target is saved as `output/target_guidesep.wav`, and the remainder of the mixture is saved as `output/residual_guidesep.wav`. This baseline uses only the waveform guide; both mel-mask conditions are disabled.


In [ ]:
from pathlib import Path
import sys
import urllib.request

UPSTREAM_COMMIT = "f6bcbdee55b56909a6dc4096f8f75fb93b47c202"
RUNTIME_ROOT = Path(".runtime")
UPSTREAM_FILES = [
    "src/models/backbones/utils.py",
    "src/models/backbones/operator_utils.py",
    "src/models/backbones/layer_utils.py",
    "src/models/backbones/attention_utils.py",
    "src/models/backbones/conditioner.py",
    "src/models/backbones/unet2d.py",
    "src/models/components/utils.py",
    "src/models/components/diffusion.py",
    "src/models/components/sampler.py",
    "src/models/components/scheduler.py",
]

for package in ["src", "src/models", "src/models/backbones", "src/models/components"]:
    package_dir = RUNTIME_ROOT / package
    package_dir.mkdir(parents=True, exist_ok=True)
    (package_dir / "__init__.py").touch()

for relative_path in UPSTREAM_FILES:
    destination = RUNTIME_ROOT / relative_path
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        url = f"https://raw.githubusercontent.com/YutongWen/GuideSep/{UPSTREAM_COMMIT}/{relative_path}"
        urllib.request.urlretrieve(url, destination)

runtime_path = str(RUNTIME_ROOT.resolve())
if runtime_path not in sys.path:
    sys.path.insert(0, runtime_path)

print(f"GuideSep runtime source ready: {RUNTIME_ROOT.resolve()}")


In [ ]:
import math

import numpy as np
import soundfile as sf
import torch
from scipy.signal import resample_poly
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

from src.models.backbones.unet2d import UNet2dBase
from src.models.components.diffusion import EluDiffusion
from src.models.components.sampler import DPMSampler
from src.models.components.scheduler import KarrasSchedule

SAMPLE_RATE = 16000
N_FFT = 510
HOP_LENGTH = 256
SPEC_ABS_EXPONENT = 0.2
SPEC_FACTOR = 0.6
SAMPLER_STEPS = 8
MIX_PATH = Path("wav/mix.wav")
GUIDE_PATH = Path("wav/guide.wav")
TARGET_PATH = Path("output/target_guidesep.wav")
RESIDUAL_PATH = Path("output/residual_guidesep.wav")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def spec_fwd(spec):
    if SPEC_ABS_EXPONENT != 1:
        spec = spec.abs() ** SPEC_ABS_EXPONENT * torch.exp(1j * spec.angle())
    return spec * SPEC_FACTOR


def spec_back(spec):
    spec = spec / SPEC_FACTOR
    if SPEC_ABS_EXPONENT != 1:
        spec = spec.abs() ** (1 / SPEC_ABS_EXPONENT) * torch.exp(1j * spec.angle())
    return spec


def load_audio(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing input file: {path.resolve()}")

    audio, sr = sf.read(str(path), always_2d=True, dtype="float32")
    audio = audio.mean(axis=1)

    if sr != SAMPLE_RATE:
        gcd = math.gcd(sr, SAMPLE_RATE)
        audio = resample_poly(
            audio,
            SAMPLE_RATE // gcd,
            sr // gcd,
        ).astype(np.float32, copy=False)

    waveform = torch.from_numpy(np.ascontiguousarray(audio)).unsqueeze(0)
    peak = waveform.abs().max()
    if peak > 0:
        waveform = waveform / peak
    return waveform


def build_net():
    net = UNet2dBase(
        cond_on_text=True,
        text_embed_dim=768,
        max_text_len=120,
        dim=128,
        cond_drop_prob=0.0,
        dim_mults=[1, 2, 2, 2],
        channels=2,
        condition_channels=5,
        num_resnet_blocks=2,
        resnet_groups=8,
        layer_attns=[False, False, True, True],
        layer_cross_attns=[False, False, True, True],
        attn_heads=2,
        ff_mult=2.0,
        memory_efficient=True,
        use_condition_block=True,
        mixture_input=False,
    )
    ckpt_path = hf_hub_download(repo_id="YutongCooper/GuideSep-v1", filename="last.ckpt")
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = checkpoint.get("state_dict", checkpoint)
    net_state = {key[4:]: value for key, value in state_dict.items() if key.startswith("net.")}
    if not net_state:
        raise RuntimeError("The checkpoint does not contain the expected 'net.' state_dict keys.")
    missing, unexpected = net.load_state_dict(net_state, strict=False)
    missing = [key for key in missing if not key.startswith("text_conditioner.")]
    if missing or unexpected:
        raise RuntimeError(f"Checkpoint/network mismatch. Missing: {missing}; unexpected: {unexpected}")
    return net.to(DEVICE).eval()


net = build_net()
diffusion = EluDiffusion(sigma_data=0.2, dynamic_threshold=0.0).to(DEVICE)
sampler = DPMSampler(num_steps=SAMPLER_STEPS, cond_scale=1, order=1, multisteps=True, x0_pred=False, log_time_spacing=False).to(DEVICE)
noise_schedule = KarrasSchedule(sigma_min=0.002, sigma_max=80.0, rho=7.0, num_steps=SAMPLER_STEPS)().to(DEVICE)
window = torch.hann_window(N_FFT, periodic=True, device=DEVICE)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Model loaded.")


In [ ]:
mixture = load_audio(MIX_PATH).to(DEVICE)
guide = load_audio(GUIDE_PATH).to(DEVICE)
common_samples = min(mixture.shape[-1], guide.shape[-1])
mixture = mixture[..., :common_samples]
guide = guide[..., :common_samples]

stft_kwargs = dict(n_fft=N_FFT, hop_length=HOP_LENGTH, center=True, normalized=True, return_complex=True)
spec_mix_raw = torch.stft(mixture, window=window, **stft_kwargs)
spec_guide_raw = torch.stft(guide, window=window, **stft_kwargs)
time_length = min(spec_mix_raw.shape[-1], spec_guide_raw.shape[-1]) // 16 * 16
if time_length < 16:
    raise ValueError("Input audio is too short for GuideSep inference.")

spec_mix_raw = spec_mix_raw[..., :time_length]
spec_guide_raw = spec_guide_raw[..., :time_length]
spec_mix = spec_fwd(spec_mix_raw).unsqueeze(1)
spec_mix = torch.cat((spec_mix.real, spec_mix.imag), dim=1)
spec_guide = spec_fwd(spec_guide_raw).unsqueeze(1).abs()

mask_channels = torch.full(
    (mixture.shape[0], 2, N_FFT // 2 + 1, time_length),
    -1.0,
    dtype=spec_mix.dtype,
    device=DEVICE,
)
condition = torch.cat((spec_mix, spec_guide, mask_channels), dim=1)
noise = torch.randn((mixture.shape[0], 2, N_FFT // 2 + 1, time_length), device=DEVICE)

with torch.inference_mode():
    predicted = sampler(
        noise,
        fn=diffusion.denoise_fn,
        net=net,
        sigmas=noise_schedule,
        inj_channels=condition,
        mixture=spec_mix,
        mask=None,
        source=None,
    )

predicted = torch.permute(predicted, (0, 2, 3, 1)).contiguous()
predicted_complex = torch.view_as_complex(predicted)[:, None, :, :]
predicted_complex = spec_back(predicted_complex)
target = torch.istft(
    predicted_complex.squeeze(1),
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    center=True,
    normalized=True,
    window=window,
).detach().cpu().squeeze(0)

mixture_reference = mixture.detach().cpu().squeeze(0)[:target.numel()]
residual = mixture_reference - target

TARGET_PATH.parent.mkdir(parents=True, exist_ok=True)
sf.write(str(TARGET_PATH), target.numpy(), SAMPLE_RATE, subtype="FLOAT")
sf.write(str(RESIDUAL_PATH), residual.numpy(), SAMPLE_RATE, subtype="FLOAT")

print("Saved target:", TARGET_PATH.resolve())
print("Saved residual:", RESIDUAL_PATH.resolve())
print(f"Output duration: {target.numel() / SAMPLE_RATE:.3f} s")
print(f"Reconstruction max error: {(mixture_reference - (target + residual)).abs().max().item():.3e}")

print("Target")
display(Audio(target.numpy(), rate=SAMPLE_RATE))
print("Residual")
display(Audio(residual.numpy(), rate=SAMPLE_RATE))
